In [0]:
import mlflow
import time
import json
from pyspark.sql import functions as F


In [0]:
# Imports and configuration
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"

exec(open(f"{UTILS_DIR}/config.py").read())
exec(open(f"{UTILS_DIR}/states.py").read())
exec(open(f"{UTILS_DIR}/llm_utils.py").read())

print("✅ Utils loaded")


In [0]:
df_extracted = spark.table(LLM_EXTRACTED).filter(
    F.col("llm_status") == "success"
)

print(f"Records to judge: {df_extracted.count()}")
display(df_extracted.select("product_id", "extracted_name", "extracted_brand", "extracted_sub_category"))

In [0]:

def judge_product(name: str, brand: str, sub_category: str) -> dict:
    """
    TechMart-specific judge logic.
    Validates extracted product info against approved taxonomy.
    Builds messages, calls LLM, validates with Pydantic.
    """
    # Load prompts for this flow
    system_prompt = load_prompt(
        PROMPTS_DIR, JUDGE_TEMPLATE, role="system",
        approved_taxonomy=APPROVED_TAXONOMY
    )
    user_prompt = load_prompt(
        PROMPTS_DIR, JUDGE_TEMPLATE, role="user",
        name=name,
        brand=brand,
        sub_category=sub_category
    )

    # Call generic LLM utility
    result = call_llm(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        api_key    = LLM_API_KEY,
        api_url    = LLM_API_URL,
        model      = LLM_MODEL,
        max_tokens = 100
    )

    # Clean and validate response with Pydantic
    response_clean = (
        result["response_text"]
        .strip()
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )
    judge_result = JudgeResult(**json.loads(response_clean))

    return {
        "approved"     : judge_result.approved,
        "reason"       : judge_result.reason,
        "input_tokens" : result["input_tokens"],
        "output_tokens": result["output_tokens"],
        "latency"      : result["latency_seconds"]
    }

In [0]:

mlflow.set_experiment(MLFLOW_EXPERIMENT)

records     = df_extracted.collect()
all_records = []

with mlflow.start_run(run_name=f"judge_{PROMPT_VERSION}"):

    # Log judge prompt template as artifact
    judge_template_raw = load_prompt_template_raw(PROMPTS_DIR, PROMPT_JUDGE)
    mlflow.log_text(judge_template_raw, PROMPT_JUDGE)

    # Log run parameters
    mlflow.log_param("prompt_version", PROMPT_VERSION)
    mlflow.log_param("model",          LLM_MODEL)
    mlflow.log_param("provider",       LLM_PROVIDER)
    mlflow.log_param("total_records",  len(records))

    approved_count   = 0
    flagged_count    = 0
    total_input_tok  = 0
    total_output_tok = 0
    total_latency    = 0.0

    for row in records:
        product_id   = row["product_id"]
        name         = row["extracted_name"]
        brand        = row["extracted_brand"]
        sub_category = row["extracted_sub_category"]

        try:
            result = judge_product(name, brand, sub_category)

            # Accumulate metrics for MLflow
            total_input_tok  += result["input_tokens"]
            total_output_tok += result["output_tokens"]
            total_latency    += result["latency"]

            if result["approved"]:
                approved_count += 1
            else:
                flagged_count += 1

            all_records.append({
                "product_id"            : str(product_id),
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_approved"        : str(result["approved"]),
                "judge_reason"          : result["reason"],
                "prompt_version"        : PROMPT_VERSION
            })

        except Exception as e:
            flagged_count += 1
            all_records.append({
                "product_id"            : str(product_id),
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_approved"        : "False",
                "judge_reason"          : f"Judge call failed: {str(e)}",
                "prompt_version"        : PROMPT_VERSION
            })

        time.sleep(0.3)

    # Log all metrics to MLflow
    mlflow.log_metric("approved_count",      approved_count)
    mlflow.log_metric("flagged_count",       flagged_count)
    mlflow.log_metric("approval_rate",       round(approved_count / len(records), 3))
    mlflow.log_metric("total_input_tokens",  total_input_tok)
    mlflow.log_metric("total_output_tokens", total_output_tok)
    mlflow.log_metric("total_tokens",        total_input_tok + total_output_tok)
    mlflow.log_metric("avg_latency_seconds", round(total_latency / len(records), 3))

print(f"✅ Judging complete: {approved_count} approved, {flagged_count} flagged")

In [0]:
df_taxonomy = spark.createDataFrame(all_records)

(df_taxonomy.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TAXONOMY))

print(f"✅ Written: {SILVER_TAXONOMY}")
display(df_taxonomy)

In [0]:
print("=== VALIDATION ===")
print(f"Total records : {spark.table(SILVER_TAXONOMY).count()}")
print(f"Approved      : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'True').count()}")
print(f"Flagged       : {spark.table(SILVER_TAXONOMY).filter(F.col('judge_approved') == 'False').count()}")

display(spark.table(SILVER_TAXONOMY))